In [ ]:
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import gaussian_kde

In [ ]:
bucket = "carbonplan-srm"
key = "output/v0.1/example_SouthAfrica/pr_model_hist_BCSD_nonparametric.nc"

BCSD = xr.open_dataset(f"s3://{bucket}/{key}")["pr"]

BCSD = BCSD.drop_vars([v for v in ["dayofyear", "ensemble_member"] if v in BCSD.coords])
times = pd.to_datetime(BCSD.time.values.astype(str))
unique_times, index = np.unique(times, return_index=True)
BCSD = BCSD.isel(time=np.sort(index))
BCSD["time"] = unique_times
BCSD = BCSD.sel(latitude=slice(-22, -35), longitude=slice(16, 33))
BCSD = BCSD * 86400.0
BCSD.attrs["units"] = "mm/day"

In [ ]:
storage = icechunk.s3_storage(
    bucket="carbonplan-srm", prefix="input/tensor/era5_rechunked_resampled.icechunk", from_env=True
)
repo = icechunk.Repository.open(storage)
session = repo.writable_session("main")
era5 = xr.open_zarr(session.store, consolidated=False)

ERA5 = era5["mean_total_precipitation_rate"]
ERA5 = ERA5.sel(time=slice("1978-01-01", "2014-12-31"))
ERA5 = ERA5.sel(latitude=slice(-22, -35), longitude=slice(16, 33))
ERA5 = ERA5 * 86400.0
ERA5.attrs["units"] = "mm/day"

In [ ]:
storage = icechunk.s3_storage(
    bucket="carbonplan-srm",
    prefix="input/tensor/CESM2-WACCM-Historical/icechunk/icechunk",
    from_env=True,
)

repo = icechunk.Repository.open(storage)
session = repo.writable_session("main")

ds_hist = xr.open_zarr(session.store, consolidated=False)
HIST = ds_hist["PRECT"]

times = pd.to_datetime(HIST.time.values.astype(str))
unique_times, index = np.unique(times, return_index=True)
HIST = HIST.isel(time=np.sort(index))
HIST["time"] = unique_times
HIST = HIST.sel(time=slice("1978-01-01", "2014-12-31"))
HIST = HIST.sel(lat=slice(-35, -22), lon=slice(16, 33))
HIST = HIST * 1000.0 * 86400.0
HIST.attrs["units"] = "mm/day"

In [ ]:
key_g6 = "output/v0.1/example_SouthAfrica/pr_g61pt5k_BCSD_nonparametric.nc"
local_g6 = key_g6.split("/")[-1]

ds_g6 = xr.open_dataset(f"s3://{bucket}/{key}")
CESM_BCSD_g61pt5k = ds_g6["pr"]

CESM_BCSD_g61pt5k = CESM_BCSD_g61pt5k.drop_vars(
    [v for v in ["dayofyear", "ensemble_member"] if v in CESM_BCSD_g61pt5k.coords], errors="ignore"
)

times = pd.to_datetime(CESM_BCSD_g61pt5k.time.values.astype(str))
unique_times, index = np.unique(times, return_index=True)
CESM_BCSD_g61pt5k = CESM_BCSD_g61pt5k.isel(time=np.sort(index))
CESM_BCSD_g61pt5k["time"] = unique_times

CESM_BCSD_g61pt5k = CESM_BCSD_g61pt5k.sel(latitude=slice(-22, -35), longitude=slice(16, 33))

CESM_BCSD_g61pt5k = CESM_BCSD_g61pt5k * 86400.0
CESM_BCSD_g61pt5k.attrs["units"] = "mm/day"

In [ ]:
bucket = "carbonplan-srm"
key_ssp = "output/v0.1/example_SouthAfrica/pr_ssp245_BCSD_nonparametric.nc"

ds_ssp = xr.open_dataset(f"s3://{bucket}/{key}")
CESM_BCSD_ssp245 = ds_ssp["pr"]

CESM_BCSD_ssp245 = CESM_BCSD_ssp245.drop_vars(
    [v for v in ["dayofyear", "ensemble_member"] if v in CESM_BCSD_ssp245.coords], errors="ignore"
)

times = pd.to_datetime(CESM_BCSD_ssp245.time.values.astype(str))
unique_times, index = np.unique(times, return_index=True)
CESM_BCSD_ssp245 = CESM_BCSD_ssp245.isel(time=np.sort(index))
CESM_BCSD_ssp245["time"] = unique_times

CESM_BCSD_ssp245 = CESM_BCSD_ssp245.sel(latitude=slice(-22, -35), longitude=slice(16, 33))

CESM_BCSD_ssp245 = CESM_BCSD_ssp245 * 86400.0
CESM_BCSD_ssp245.attrs["units"] = "mm/day"

In [ ]:
all_datasets = {
    "CESM_BCSD_g61pt5k": CESM_BCSD_g61pt5k,
    "CESM_BCSD_ssp245": CESM_BCSD_ssp245,
    "CESM_BCSD_hist": BCSD,
    "CESM_hist_raw": HIST,
    "ERA5": ERA5,
}

print("Percent of negative precipitation values v0.1:\n")

for name, da in all_datasets.items():
    neg = (da < 0).sum().compute()
    total = da.size
    pct = float(neg) / total * 100
    print(f"{name:20s}: {pct:.6f}%")

In [ ]:
g6 = CESM_BCSD_g61pt5k
era5 = ERA5

g6_vals = g6.values.flatten()
era_vals = era5.values.flatten()

g6_vals = g6_vals[np.isfinite(g6_vals)]
era_vals = era_vals[np.isfinite(era_vals)]

xmin = min(g6_vals.min(), era_vals.min())
xmax = max(np.percentile(g6_vals, 99.9), np.percentile(era_vals, 99.9))

bins = np.linspace(xmin, xmax, 200)

plt.figure(figsize=(8, 5))

plt.hist(g6_vals, bins=bins, histtype="step", linewidth=2, color="blue", label="CESM_BCSD_g61pt5k")

plt.hist(era_vals, bins=bins, histtype="step", linewidth=2, color="red", label="ERA5")

plt.ylim(0, 1_000_000)
plt.xlabel("Precipitation value (mm/day)")
plt.ylabel("Count")
plt.title("Histogram of South Africa Precipitation Values v0.1")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

In [ ]:
lat_ct, lon_ct = -33.9, 18.4

bcsd_ct = BCSD.sel(latitude=lat_ct, longitude=lon_ct, method="nearest")
era_ct = ERA5.sel(latitude=lat_ct, longitude=lon_ct, method="nearest")
hist_ct = HIST.sel(lat=lat_ct, lon=lon_ct, method="nearest")

In [ ]:
h = hist_ct.values.astype(float).flatten()
e = era_ct.values.astype(float).flatten()
b = bcsd_ct.values.astype(float).flatten()

h = h[np.isfinite(h)]
e = e[np.isfinite(e)]
b = b[np.isfinite(b)]

all_vals = np.concatenate([h, e, b])
p95 = np.percentile(all_vals, 95)

h_hi = h[h >= p95]
e_hi = e[e >= p95]
b_hi = b[b >= p95]

kde_h = gaussian_kde(h_hi)
kde_e = gaussian_kde(e_hi)
kde_b = gaussian_kde(b_hi)

xmin = p95
xmax = np.percentile(all_vals, 99.9)
xgrid = np.linspace(xmin, xmax, 500)

plt.figure(figsize=(8, 5))

plt.plot(xgrid, kde_h(xgrid), linewidth=2.0, label="CESM_hist")
plt.plot(xgrid, kde_e(xgrid), linewidth=2.0, label="ERA5")
plt.plot(xgrid, kde_b(xgrid), linewidth=2.0, label="BCSD_hist")

plt.xlabel("Daily precipitation (mm/day)")
plt.ylabel("KDE Density")
plt.title("KDE of Top 5% Precipitation for historical Cape Town v0.1")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
storage_ssp = icechunk.s3_storage(
    bucket="carbonplan-srm",
    prefix="input/tensor/CESM2-WACCM-SSP245/icechunk/icechunk",
    from_env=True,
)

repo_ssp = icechunk.Repository.open(storage_ssp)
session_ssp = repo_ssp.writable_session("main")

ds_ssp = xr.open_zarr(session_ssp.store, consolidated=False)
SSP = ds_ssp["PRECT"]

times = pd.to_datetime(SSP.time.values.astype(str))
unique_times, index = np.unique(times, return_index=True)
SSP = SSP.isel(time=np.sort(index))
SSP["time"] = unique_times

SSP = SSP.sel(time=slice("2050-01-01", "2069-12-31"))
SSP = SSP.sel(lat=slice(-35, -22), lon=slice(16, 33))

SSP = SSP * 1000.0 * 86400.0
SSP.attrs["units"] = "mm/day"
SSP = SSP.sel(ensemble_member="006")

In [ ]:
storage_g61p5 = icechunk.s3_storage(
    bucket="carbonplan-srm", prefix="input/tensor/CESM-G6-1.5K/icechunk/icechunk", from_env=True
)

repo_g61p5 = icechunk.Repository.open(storage_g61p5)
session_g61p5 = repo_g61p5.writable_session("main")

ds_g61p5 = xr.open_zarr(session_g61p5.store, consolidated=False)
G6_15 = ds_g61p5["PRECT"]

times = pd.to_datetime(G6_15.time.values.astype(str))
unique_times, index = np.unique(times, return_index=True)
G6_15 = G6_15.isel(time=np.sort(index))
G6_15["time"] = unique_times


G6_15 = G6_15.sel(time=slice("2050-01-01", "2069-12-31"))
G6_15 = G6_15.sel(lat=slice(-35, -22), lon=slice(16, 33))

G6_15 = G6_15 * 1000.0 * 86400.0
G6_15.attrs["units"] = "mm/day"

In [ ]:
bucket = "carbonplan-srm"
key = "output/v0.1/example_SouthAfrica/pr_ssp245_BCSD_nonparametric.nc"
BCSD_SSP = xr.open_dataset(f"s3://{bucket}/{key}")["pr"]

In [ ]:
bucket = "carbonplan-srm"
key = "output/v0.1/example_SouthAfrica/pr_g61pt5k_BCSD_nonparametric.nc"
BCSD_G6_15 = xr.open_dataset(f"s3://{bucket}/{key}")["pr"]

In [ ]:
lat_ct, lon_ct = -33.9, 18.4

BCSD_G6_15_ct = BCSD_G6_15.sel(latitude=lat_ct, longitude=lon_ct, method="nearest")
BCSD_SSP_ct = BCSD_SSP.sel(latitude=lat_ct, longitude=lon_ct, method="nearest")
G6_15_ct = G6_15.sel(lat=lat_ct, lon=lon_ct, method="nearest")
SSP_ct = SSP.sel(lat=lat_ct, lon=lon_ct, method="nearest")

In [ ]:
BCSD_SSP_ct = BCSD_SSP_ct.where(BCSD_SSP_ct.ensemble_member == "006", drop=True)
BCSD_G6_15_ct = BCSD_G6_15_ct.where(BCSD_G6_15_ct.ensemble_member == "006", drop=True)

In [ ]:
BCSD_SSP_ct_mm = BCSD_SSP_ct.values * 86400.0
BCSD_G6_15_ct_mm = BCSD_G6_15_ct.values * 86400.0

G6_15_ct_mm = G6_15_ct.values
SSP_ct_mm = SSP_ct.values

datasets = {
    "BCSD SSP245": BCSD_SSP_ct_mm.flatten(),
    "BCSD G6-1.5K": BCSD_G6_15_ct_mm.flatten(),
    "CESM SSP245": SSP_ct_mm.flatten(),
    "CESM G6-1.5K": G6_15_ct_mm.flatten(),
}

for k in datasets:
    d = datasets[k]
    datasets[k] = d[np.isfinite(d)]

all_vals = np.concatenate(list(datasets.values()))
p95 = np.percentile(all_vals, 95)

kdes = {}
for name, arr in datasets.items():
    tail = arr[arr >= p95]
    if tail.size >= 5:
        kdes[name] = gaussian_kde(tail)

xmin = p95
xmax = np.percentile(all_vals, 99.9)
xgrid = np.linspace(xmin, xmax, 500)

plt.figure(figsize=(8, 5))

colors = {
    "BCSD SSP245": "tab:blue",
    "BCSD G6-1.5K": "tab:green",
    "CESM SSP245": "tab:red",
    "CESM G6-1.5K": "tab:purple",
}

for name, kde in kdes.items():
    plt.plot(xgrid, kde(xgrid), linewidth=2.0, label=name, color=colors[name])

plt.xlabel("Daily precipitation (mm/day)")
plt.ylabel("KDE density")
plt.title("KDE of Top 5% Precipitation for future Cape Town v0.1")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()